In [1]:
!pip install experta
!pip install frozendict==1.2

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping

from experta import KnowledgeEngine, Rule, Fact, MATCH, AS, P, NOT, AND, OR

class Email(Fact):
    """Representa a estrutura de um e-mail bruto recebido pelo sistema."""
    pass

class Extracao(Fact):
    """NÍVEL 1: Fatos intermediários extraídos do texto e metadados."""
    pass

class Classificacao(Fact):
    """NÍVEL 2: Categorização lógica baseada nas extrações."""
    pass

class DecisaoFinal(Fact):
    """NÍVEL 3: Ação final recomendada pelo motor com a justificativa."""
    pass

class MotorClassificadorEmails(KnowledgeEngine):
    
    def inicializar_trace(self):
        self.trace_decisoes = []

    def adicionar_log(self, regra, motivo):
        log_msg = f"[{regra}] disparada -> {motivo}"
        self.trace_decisoes.append(log_msg)
        print(log_msg)

    # NÍVEL 1
    @Rule(Email(id=MATCH.id, corpo=P(lambda c: any(w in c.lower() for w in ["ganhou", "loteria", "herança", "link suspeito"]))))
    def R1_detecta_phishing(self, id):
        self.declare(Extracao(email_id=id, propriedade="phishing", indicador="conteudo_malicioso"))
        self.adicionar_log("R1_detecta_phishing", f"E-mail {id} contém termos altamente suspeitos no corpo.")

    @Rule(Email(id=MATCH.id, remetente=P(lambda r: r.endswith("ufpb.br"))))
    def R2_detecta_institucional(self, id):
        self.declare(Extracao(email_id=id, propriedade="institucional", indicador="dominio_confiavel"))
        self.adicionar_log("R2_detecta_institucional", f"E-mail {id} originou-se de um domínio institucional confiável.")

    @Rule(Email(id=MATCH.id, assunto=P(lambda a: any(w in a.lower() for w in ["urgente", "prazo", "imediato", "importante"]))))
    def R3_detecta_urgencia(self, id):
        self.declare(Extracao(email_id=id, propriedade="urgente", indicador="marcador_temporal"))
        self.adicionar_log("R3_detecta_urgencia", f"E-mail {id} possui palavras de alta prioridade temporal no assunto.")

    @Rule(Email(id=MATCH.id, assunto=P(lambda a: any(w in a.lower() for w in ["desconto", "oferta", "compre", "promoção"]))))
    def R4_detecta_comercial(self, id):
        self.declare(Extracao(email_id=id, propriedade="comercial", indicador="marketing_direto"))
        self.adicionar_log("R4_detecta_comercial", f"E-mail {id} possui gatilhos comerciais no assunto.")

    # NÍVEL 2
    @Rule(Extracao(email_id=MATCH.id, propriedade="phishing"), salience=100)
    def R5_classifica_spam(self, id):
        self.declare(Classificacao(email_id=id, categoria="spam", prioridade="critica"))
        self.adicionar_log("R5_classifica_spam", f"E-mail {id} foi categorizado estritamente como SPAM devido a indicadores de fraude.")

    @Rule(AND(
        Extracao(email_id=MATCH.id, propriedade="institucional"),
        Extracao(email_id=MATCH.id, propriedade="urgente")
    ))
    def R6_classifica_trabalho_urgente(self, id):
        self.declare(Classificacao(email_id=id, categoria="trabalho_urgente", prioridade="alta"))
        self.adicionar_log("R6_classifica_trabalho_urgente", f"E-mail {id} categorizado como TRABALHO CRÍTICO (Origem confiável + Urgência).")

    @Rule(AND(
        Extracao(email_id=MATCH.id, propriedade="institucional"),
        NOT(Extracao(email_id=MATCH.id, propriedade="urgente"))
    ))
    def R7_classifica_trabalho_comum(self, id):
        self.declare(Classificacao(email_id=id, categoria="trabalho_rotina", prioridade="media"))
        self.adicionar_log("R7_classifica_trabalho_comum", f"E-mail {id} categorizado como TRABALHO ROTINA (Institucional sem marcadores de urgência).")

    @Rule(AND(
        Extracao(email_id=MATCH.id, propriedade="comercial"),
        NOT(Extracao(email_id=MATCH.id, propriedade="phishing"))
    ))
    def R8_classifica_promocoes(self, id):
        self.declare(Classificacao(email_id=id, categoria="promocoes", prioridade="baixa"))
        self.adicionar_log("R8_classifica_promocoes", f"E-mail {id} categorizado como PROMOÇÕES (Conteúdo comercial limpo).")

    # NÍVEL 3
    @Rule(Classificacao(email_id=MATCH.id, categoria="spam"))
    def R9_acao_quarentena(self, id):
        self.declare(DecisaoFinal(email_id=id, acao="MOVER_PARA_QUARENTENA", motivo="Bloqueado preventivamente por risco de segurança."))
        self.adicionar_log("R9_acao_quarentena", f"Decisão final para E-mail {id}: Direcionado para a Quarentena.")

    @Rule(Classificacao(email_id=MATCH.id, categoria="trabalho_urgente"))
    def R10_acao_notificacao_imediata(self, id):
        self.declare(DecisaoFinal(email_id=id, acao="ENTRADA_PRIORITARIA_E_PUSH", motivo="Notificar usuário imediatamente via SMS/Push."))
        self.adicionar_log("R10_acao_notificacao_imediata", f"Decisão final para E-mail {id}: Movido para Caixa Prioritária com Alerta.")

    @Rule(Classificacao(email_id=MATCH.id, categoria="trabalho_rotina"))
    def R11_acao_caixa_geral(self, id):
        self.declare(DecisaoFinal(email_id=id, acao="MOVER_PARA_TRABALHO", motivo="Arquivar na pasta corporativa padrão."))
        self.adicionar_log("R11_acao_caixa_geral", f"Decisão final para E-mail {id}: Movido para Caixa de Trabalho Comum.")

    @Rule(Classificacao(email_id=MATCH.id, categoria="promocoes"))
    def R12_acao_aba_marketing(self, id):
        self.declare(DecisaoFinal(email_id=id, acao="MOVER_PARA_ABA_PROMOCOES", motivo="Desviar da caixa principal para leitura posterior."))
        self.adicionar_log("R12_acao_aba_marketing", f"Decisão final para E-mail {id}: Direcionado para a aba de Promoções.")